# Capítulo 5 · SVM: clasificar y predecir un número

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/lectures/chapter_05/07_svm_clasificacion_y_regresion.ipynb)

**Aprendizaje supervisado · Uso de SVM**

SVM también sirve para las dos tareas.

- **Clasificación (`SVC`):** busca la frontera con el margen más ancho entre las clases. Los puntos que quedan justo en el borde del margen son los vectores de soporte.
- **Regresión (`SVR`):** ajusta una función y tolera un tubo a su alrededor. Lo que cae dentro del tubo no cuenta como error.

`kernel="linear"` es una frontera recta (o un plano). `kernel="rbf"` puede doblarla. Como KNN, SVM usa las distancias de los números: si una columna va por miles y otra por decenas, hay que estandarizar antes.

### 📚 Índice
1. Iris: un vistazo y clasificar
2. Autos: un vistazo y predecir millas por galón
3. Chuleta resumen


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR

sns.set_theme(style="whitegrid")
palette = sns.color_palette("deep")

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = raiz / "data"


---
## 1. Iris: un vistazo y clasificar

Las mismas 150 flores. Con el largo y el ancho del pétalo ya se ve por qué una frontera puede separar las especies.


In [ ]:
iris = pd.read_csv(DATA_DIR / "iris.csv")
especies = ["setosa", "versicolor", "virginica"]
iris.head()


### Cómo se ve

50 flores de cada especie. La setosa queda aparte en el pétalo; versicolor y virginica se tocan. El largo y el ancho del pétalo van casi juntos.


In [ ]:
iris["target"].map(dict(enumerate(especies))).value_counts().rename("flores")


In [ ]:
iris.drop(columns="target").describe()


In [ ]:
iris_plot = iris.assign(especie=iris["target"].map(dict(enumerate(especies))))
sns.scatterplot(
    data=iris_plot, x="petal length (cm)", y="petal width (cm)",
    hue="especie", palette=palette[:3]
)
plt.show()


In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(
    iris.drop(columns="target").corr(), annot=True, fmt=".2f",
    cmap="vlag", center=0, vmin=-1, vmax=1
)
plt.show()


Con solo el pétalo, un `SVC` lineal dibuja la frontera. El color de fondo es la clase que predice. `C` dice cuánto se castiga un punto del lado equivocado: más grande, margen más estricto.


In [ ]:
X_petalo = iris[["petal length (cm)", "petal width (cm)"]]
y = iris["target"]
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_petalo, y, test_size=0.2, random_state=42, stratify=y
)

svm_lineal = Pipeline([
    ("escala", StandardScaler()),
    ("svm", SVC(kernel="linear", C=1)),
])
svm_lineal.fit(X_train_p, y_train_p)

largo = np.linspace(X_petalo.iloc[:, 0].min() - 0.3, X_petalo.iloc[:, 0].max() + 0.3, 200)
ancho = np.linspace(X_petalo.iloc[:, 1].min() - 0.3, X_petalo.iloc[:, 1].max() + 0.3, 200)
xx, yy = np.meshgrid(largo, ancho)
malla = pd.DataFrame({
    "petal length (cm)": xx.ravel(),
    "petal width (cm)": yy.ravel(),
})
zz = svm_lineal.predict(malla).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, zz, alpha=0.25, cmap="Set2")
sns.scatterplot(
    data=iris_plot, x="petal length (cm)", y="petal width (cm)",
    hue="especie", palette=palette[:3]
)
plt.title(f"Accuracy en test: {svm_lineal.score(X_test_p, y_test_p):.2f}")
plt.show()


Con las 4 medidas, el lineal y el `rbf` quedan parecidos en este dataset: las especies ya se separan bastante. Los dos van dentro de un `Pipeline` con `StandardScaler`.


In [ ]:
X = iris.drop(columns="target")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

filas = []
for kernel in ["linear", "rbf"]:
    modelo = Pipeline([
        ("escala", StandardScaler()),
        ("svm", SVC(kernel=kernel, C=1)),
    ])
    modelo.fit(X_train, y_train)
    filas.append({
        "kernel": kernel,
        "accuracy_train": modelo.score(X_train, y_train),
        "accuracy_test": modelo.score(X_test, y_test),
    })

pd.DataFrame(filas).round(3)


### Elegir `C` y el kernel

Definimos el modelo base, la rejilla de valores y las particiones por separado. `GridSearchCV` prueba las combinaciones de `C` y `kernel` y elige la mayor accuracy media en validación cruzada.

El `Pipeline` une el escalado y SVM para ajustar `StandardScaler` solo con los datos de entrenamiento de cada partición. En la rejilla, `svm__C` indica el parámetro `C` del paso llamado `svm`.


In [ ]:
modelo_base = Pipeline([
    ("escala", StandardScaler()),
    ("svm", SVC()),
])

rejilla = {
    "svm__C": [0.1, 1, 10],
    "svm__kernel": ["linear", "rbf"],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

busqueda = GridSearchCV(modelo_base, rejilla, cv=cv, scoring="accuracy")
busqueda.fit(X_train, y_train)
print(busqueda.best_params_)
print(f"Accuracy CV:   {busqueda.best_score_:.3f}")
print(f"Accuracy test: {busqueda.score(X_test, y_test):.3f}")


### Reporte de clasificación y matriz de confusión

La accuracy resume los aciertos; el reporte permite revisar cada especie usando las predicciones del modelo elegido por validación cruzada sobre **test**.

- **Precision:** de las flores predichas como una especie, qué proporción era de esa especie.
- **Recall:** de las flores que realmente eran de una especie, qué proporción encontramos.
- **F1-score:** combina precision y recall mediante su media armónica.
- **Support:** cuántas flores reales hay de cada especie en test.

`macro avg` da el mismo peso a cada especie; `weighted avg` pondera por su cantidad de flores.
En la matriz, las filas son clases reales y las columnas son predicciones. La diagonal contiene los aciertos; fuera de ella aparecen las confusiones. Test sirve para evaluar, no para seguir ajustando hiperparámetros.


In [ ]:
y_pred = busqueda.predict(X_test)
print(classification_report(
    y_test, y_pred, labels=[0, 1, 2], target_names=especies,
    digits=3, zero_division=0,
))


In [ ]:
matriz = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
sns.heatmap(
    matriz, annot=True, fmt="d", cmap="Blues",
    xticklabels=especies, yticklabels=especies,
)
plt.title("SVM · Matriz de confusión en test")
plt.xlabel("Especie predicha")
plt.ylabel("Especie real")
plt.tight_layout()
plt.show()


---
## 2. Autos: un vistazo y predecir millas por galón

`SVR` usa el mismo esquema. `kernel="linear"` sale casi una recta; `rbf` puede seguir la curva del peso.


### Cómo se ve

398 autos y 6 sin `horsepower`. A más peso, menos millas por galón. El peso está en miles y el año entre 70 y 82: sin estandarizar, el peso se queda con la frontera.


In [ ]:
autos = pd.read_csv(DATA_DIR / "mpg.csv")
autos.head()


In [ ]:
autos.isna().sum().rename("nulos")


In [ ]:
autos.describe()


In [ ]:
sns.histplot(data=autos, x="mpg", bins=20, color=palette[0])
plt.show()


In [ ]:
sns.scatterplot(data=autos, x="weight", y="mpg", color=palette[0], alpha=0.7)
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(autos.corr(numeric_only=True), annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1)
plt.show()


In [ ]:
autos = autos.dropna()
X_peso = autos[["weight"]]
y_mpg = autos["mpg"]
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_peso, y_mpg, test_size=0.2, random_state=42
)

pesos = np.linspace(autos["weight"].min(), autos["weight"].max(), 300)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, kernel in zip(axes, ["linear", "rbf"]):
    modelo = Pipeline([
        ("escala", StandardScaler()),
        ("svm", SVR(kernel=kernel, C=10)),
    ])
    modelo.fit(X_train_w, y_train_w)
    curva = modelo.predict(pd.DataFrame({"weight": pesos}))
    sns.scatterplot(data=autos, x="weight", y="mpg", color=palette[0], alpha=0.35, ax=ax)
    ax.plot(pesos, curva, color=palette[1], linewidth=2)
    ax.set_title(kernel)
plt.show()


Con peso, potencia y año, la misma `SVR` sin escalar se queda corta. Escalada, el `rbf` aprovecha las tres columnas; el lineal no puede doblar la relación.


In [ ]:
columnas = ["weight", "horsepower", "model_year"]
X_auto = autos[columnas]
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_auto, y_mpg, test_size=0.2, random_state=42
)

filas = []
for nombre, pasos in {
    "rbf sin escalar": [("svm", SVR(kernel="rbf", C=10))],
    "lineal escalado": [("escala", StandardScaler()), ("svm", SVR(kernel="linear", C=10))],
    "rbf escalado": [("escala", StandardScaler()), ("svm", SVR(kernel="rbf", C=10))],
}.items():
    modelo = Pipeline(pasos)
    modelo.fit(X_train_a, y_train_a)
    pred = modelo.predict(X_test_a)
    filas.append({
        "modelo": nombre,
        "R2_train": modelo.score(X_train_a, y_train_a),
        "R2_test": modelo.score(X_test_a, y_test_a),
        "RMSE_test": root_mean_squared_error(y_test_a, pred),
    })

pd.DataFrame(filas).round(3)


Repetimos el esquema de modelo base, rejilla y particiones para `SVR`. Probamos `C`, `kernel` y `epsilon` (el ancho del tubo). El criterio es el $R^2$ de validación cruzada.


In [ ]:
modelo_base_reg = Pipeline([
    ("escala", StandardScaler()),
    ("svm", SVR()),
])

rejilla_reg = {
    "svm__C": [1, 10, 100],
    "svm__kernel": ["linear", "rbf"],
    "svm__epsilon": [0.1, 1],
}
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

busqueda_reg = GridSearchCV(modelo_base_reg, rejilla_reg, cv=cv_reg, scoring="r2")
busqueda_reg.fit(X_train_a, y_train_a)
print(busqueda_reg.best_params_)
print(f"R² CV:   {busqueda_reg.best_score_:.3f}")
print(f"R² test: {busqueda_reg.score(X_test_a, y_test_a):.3f}")


---
## 3. 📋 Chuleta resumen

| Tarea | Código |
|---|---|
| Clasificar | `SVC(kernel="rbf", C=1)` |
| Predecir un número | `SVR(kernel="rbf", C=10)` |
| Frontera recta | `kernel="linear"` |
| Frontera curva | `kernel="rbf"` |
| Estandarizar y luego el modelo | `Pipeline([("escala", StandardScaler()), ("svm", ...)])` |
| Ancho del tubo en regresión | `SVR(epsilon=0.1)` |
| Elegir hiperparámetros | `GridSearchCV(modelo_base, rejilla, cv=cv)` |
| Train y test, clasificación | `train_test_split(..., stratify=y, random_state=42)` |
| Train y test, regresión | `train_test_split(..., random_state=42)` sin `stratify` |

> 💡 `C` grande castiga más los puntos del lado equivocado. Sin `StandardScaler`, una columna con números enormes se queda con el modelo.
